<a href="https://colab.research.google.com/github/mohdishaqedunet-cmyk/TASK-16/blob/main/TASK_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
data_df = pd.read_csv('/content/breast-cancer.csv')

# Define features (X) and target (y) for breast cancer classification
# 'diagnosis' is the target variable (M=malignant, B=benign)
# 'id' is an identifier and should not be used as a feature
X = data_df.drop(columns=['id', 'diagnosis'])
y = data_df['diagnosis'].map({'M': 1, 'B': 0}) # Encode M as 1 (malignant), B as 0 (benign)

# Check for missing values
print(X.isnull().sum().sum())   # Should be 0

0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [13]:
# Convert categorical columns into numeric using one-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Then split again
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_default = RandomForestClassifier(random_state=42)
rf_default.fit(X_train, y_train)

y_pred_default = rf_default.predict(X_test)

print("Default Model Accuracy:",
      accuracy_score(y_test, y_pred_default))


Default Model Accuracy: 0.9649122807017544


In [8]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}


In [14]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,                # 5-fold cross validation
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 20],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='accuracy')

In [15]:
print("Best Parameters:", grid_search.best_params_)

best_model = grid_search.best_estimator_


Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}


In [16]:
y_pred_tuned = best_model.predict(X_test)

print("Tuned Model Accuracy:",
      accuracy_score(y_test, y_pred_tuned))

print("\nClassification Report:\n",
      classification_report(y_test, y_pred_tuned))


Tuned Model Accuracy: 0.9649122807017544

Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.97        71
           1       0.98      0.93      0.95        43

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [17]:
default_acc = accuracy_score(y_test, y_pred_default)
tuned_acc = accuracy_score(y_test, y_pred_tuned)

print("Default Accuracy:", default_acc)
print("Tuned Accuracy:", tuned_acc)


Default Accuracy: 0.9649122807017544
Tuned Accuracy: 0.9649122807017544


In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [19]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.1, 0.01],
    'kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(
    SVC(),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.1, 0.01],
                         'kernel': ['rbf', 'linear']},
             scoring='accuracy')